# Dataset Adequacy Analysis — PEP725 Fruit Trees

Investigates whether the `PEP725_fruit_trees` dataset provides sufficient
coverage to fit a multi-species phenology model across nine fruit-tree species.

Key questions:
1. **Spatial confounding** — are species clustered in specific countries/latitudes, making species effects hard to separate from location effects?
2. **Shared-location coverage** — how many stations have observations for multiple species (essential for species-level parameter estimation)?
3. **Temporal variation** — do the observed years span meaningfully different climates?
4. **Bloom DOY range** — does each species show enough phenological variation to constrain model parameters?

Set `USE_SYNTHETIC = True` to replace the real dataset with controlled synthetic data
for debugging or to demonstrate what adequate/inadequate coverage looks like.

## Config

In [ ]:
# ── toggle real vs synthetic ───────────────────────────────────────────────
USE_SYNTHETIC        = False
DOWNLOAD_TEMPERATURE = True   # download OpenMeteo temperature; set False to skip

# Synthetic-mode settings
SYN_N_SPECIES   = 4       # number of species to simulate
SYN_N_LOCS      = 30      # locations per species
SYN_N_YEARS     = 20      # years per location
SYN_CONFOUNDED  = True    # True  → each species at disjoint latitude bands
                           # False → all species at overlapping latitudes

# ── dataset settings ──────────────────────────────────────────────────────
DATASET_KEY = 'PEP725_fruit_trees'
TARGET_OBS  = 'BBCH_60'   # first full bloom

# ── cross-factor binning ──────────────────────────────────────────────────
LAT_BINS      = [35, 40, 45, 50, 55, 60, 65]
LAT_LABELS    = ['35–40°N', '40–45°N', '45–50°N', '50–55°N', '55–60°N', '60–65°N']

# Winter temperature (Oct–Jan) bins
CLIMATE_BINS   = [-5, 0, 2, 4, 6, 8, 10, 15]
CLIMATE_LABELS = ['<0°C', '0–2°C', '2–4°C', '4–6°C', '6–8°C', '8–10°C', '>10°C']

# Spring temperature (Feb–Apr) bins
SPRING_BINS    = [0, 4, 6, 8, 10, 12, 14, 20]
SPRING_LABELS  = ['<4°C', '4–6°C', '6–8°C', '8–10°C', '10–12°C', '12–14°C', '>14°C']


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from pysephone.constants import (
    KEY_OBS_TYPE, KEY_OBSERVATIONS, KEY_SPECIES_ID, KEY_LOC_ID, KEY_YEAR,
    KEY_FEATURES,
)
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.calendar import Calendar
from pysephone.dataset.util.openmeteo import OpenMeteoFeatures
from pysephone.models.util.func_phenology import func_chilling_days


## 1. Load data

In [ ]:
if not USE_SYNTHETIC:
    obs = Dataset.load(DATASET_KEY).observations

    sp_names   = obs.species_names
    id_to_name = {sid: name for (src, sid), name in sp_names.items()}

    df_raw  = obs._df_y.copy()
    df_bloom = df_raw.xs(TARGET_OBS, level=KEY_OBS_TYPE).copy()
    df_bloom['doy'] = df_bloom[KEY_OBSERVATIONS].dt.dayofyear
    df_bloom = df_bloom.reset_index()

    locs = obs._df_y_loc.reset_index()
    df   = df_bloom.merge(locs, on=['src', 'loc_id'], how='left')
    df['species_name'] = df[KEY_SPECIES_ID].map(id_to_name)

    print(f'Loaded {DATASET_KEY}')
    print(f'  {len(df)} observations  |  '
          f'{df["loc_id"].nunique()} locations  |  '
          f'{df["year"].nunique()} years  |  '
          f'{df["species_id"].nunique()} species')
    print()
    print(df.groupby('species_name')['doy'].agg(['count', 'mean', 'std']).round(1))

else:
    rng = np.random.default_rng(0)
    lat_ranges = [
        (36, 42), (42, 47), (47, 52), (52, 57),
        (57, 62), (36, 62), (36, 62), (36, 62),
    ][:SYN_N_SPECIES]
    if not SYN_CONFOUNDED:
        lat_ranges = [(36, 62)] * SYN_N_SPECIES

    species_list = [f'Species {chr(65+i)}' for i in range(SYN_N_SPECIES)]
    countries    = ['DE', 'AT', 'CH', 'FR', 'IT', 'PL', 'CZ']
    rows = []
    for sp_id, (sp_name, (lat_lo, lat_hi)) in enumerate(
            zip(species_list, lat_ranges)):
        lats = rng.uniform(lat_lo, lat_hi, SYN_N_LOCS)
        lons = rng.uniform(5, 20, SYN_N_LOCS)
        cc   = [countries[int((lat - 35) / 5) % len(countries)] for lat in lats]
        for i, (lat, lon, country) in enumerate(zip(lats, lons, cc)):
            # Synthetic climate: colder further north
            mean_wT = 8.0 - (lat - 46) * 0.6 + rng.normal(0, 0.5)
            mean_sT = 14.0 - (lat - 46) * 0.4 + rng.normal(0, 0.5)
            chill   = 80 + (lat - 46) * 8 + rng.normal(0, 5)
            fgdu    = 1200 - (lat - 46) * 30 + rng.normal(0, 50)
            base_doy = 60 + (52 - lat) * 2 + sp_id * 5
            for year in range(2000, 2000 + SYN_N_YEARS):
                yr_anom = rng.normal(0, 3)
                doy = int(base_doy + yr_anom + rng.normal(0, 2))
                rows.append({
                    'src': 'synthetic', 'loc_id': f'{sp_id}_{i}',
                    'year': year, 'species_id': sp_id,
                    'species_name': sp_name,
                    'lat': lat, 'lon': lon, 'country_code': country,
                    'doy': doy,
                    'mean_winter_T': mean_wT + yr_anom * 0.3,
                    'mean_spring_T': mean_sT - yr_anom * 0.2,
                    'chill_days': chill - yr_anom * 2,
                    'forcing_gdu': fgdu + yr_anom * 10,
                })
    df = pd.DataFrame(rows)
    id_to_name = {i: n for i, n in enumerate(species_list)}
    print(f'Generated {len(df)} synthetic samples '
          f'({"confounded" if SYN_CONFOUNDED else "balanced"})')

# ── Derived columns ──────────────────────────────────────────────────────
df['lat_bin'] = pd.cut(df['lat'], bins=LAT_BINS, labels=LAT_LABELS)

species_order = (
    df.groupby('species_name')['doy'].median()
      .sort_values()
      .index.tolist()
)
palette = dict(zip(species_order, sns.color_palette('tab10', len(species_order))))


In [ ]:
# ── Temperature download ────────────────────────────────────────────────
# Each item covers one (loc, year, species) combination.
# Temperature features are the same for all species at the same (loc, year),
# so we deduplicate by (src, loc_id, year) before computing climate stats.

HAS_TEMP = False   # updated below if temperature data is available

if DOWNLOAD_TEMPERATURE and not USE_SYNTHETIC:
    cal      = Calendar(default_start='10-01', default_length=365)
    features = OpenMeteoFeatures(calendar=cal)
    ds = Dataset.load(DATASET_KEY, calendar=cal, feature_providers=[features])
    ds.download_features(verbose=True)

    temp_rows = {}
    for item in ds.iter_items():
        if TARGET_OBS not in item.get('observations_index', {}):
            continue
        key = (item['src'], item['loc_id'], item['year'])
        if key in temp_rows:      # same (loc,year) already processed
            continue
        ts = item[KEY_FEATURES]['temperature_2m_mean'].astype(float)
        temp_rows[key] = {
            'src':           item['src'],
            'loc_id':        item['loc_id'],
            'year':          item['year'],
            'mean_winter_T': float(ts[:120].mean()),    # Oct–Jan
            'mean_spring_T': float(ts[120:212].mean()), # Feb–Apr
            'chill_days':    float(func_chilling_days(ts).sum()),
            'forcing_gdu':   float(np.maximum(ts - 4.0, 0).sum()),
        }

    df_temp = pd.DataFrame(list(temp_rows.values()))
    df = df.merge(df_temp, on=['src', 'loc_id', 'year'], how='left')
    HAS_TEMP = True
    print(f'Temperature joined for {df["mean_winter_T"].notna().sum()} / {len(df)} rows')

elif USE_SYNTHETIC:
    # Synthetic data already has temperature columns
    HAS_TEMP = True

else:
    print('DOWNLOAD_TEMPERATURE = False — skipping temperature sections.')
    print('Set DOWNLOAD_TEMPERATURE = True and re-run to enable climate analysis.')


## 2. Spatial coverage

Map each observation station coloured by species. Spatial clustering of species into disjoint countries or latitude bands is the first visual sign of confounding.

In [ ]:
loc_df = df.groupby(['loc_id', 'species_name']).agg(
    lat=('lat', 'first'), lon=('lon', 'first'),
    n=('doy', 'count'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Station coverage by species', fontsize=12, fontweight='bold')

# — station map ——————————————————————————————————————————————————————
ax = axes[0]
for sp in species_order:
    sub = loc_df[loc_df['species_name'] == sp]
    if sub.empty:
        continue
    ax.scatter(sub['lon'], sub['lat'],
               c=[palette[sp]], s=sub['n'] * 0.5 + 10, alpha=0.6,
               label=f'{sp}  (n_loc={sub["loc_id"].nunique()})',
               edgecolors='white', linewidths=0.3)
ax.set_xlabel('Longitude', fontsize=9)
ax.set_ylabel('Latitude', fontsize=9)
ax.set_title('Station map  (marker size ∝ years of data)', fontsize=9, fontweight='bold')
ax.legend(fontsize=7.5, ncol=2, framealpha=0.9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — latitude distributions ———————————————————————————————————————————
ax = axes[1]
lat_bins = np.linspace(df['lat'].min() - 1, df['lat'].max() + 1, 30)
for sp in species_order:
    sub = df[df['species_name'] == sp]['lat']
    ax.hist(sub, bins=lat_bins, alpha=0.45, color=palette[sp],
            label=sp, edgecolor='none')
ax.set_xlabel('Latitude (°N)', fontsize=9)
ax.set_ylabel('Sample count', fontsize=9)
ax.set_title('Latitude distribution by species', fontsize=9, fontweight='bold')
ax.legend(fontsize=7.5, ncol=2, framealpha=0.9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


## 3. Cross-factor coverage

Coverage matrices showing sample counts for each (species, factor) pair.
A good dataset should have non-zero counts in most cells.
A nearly **rank-1** matrix signals that species and environment are confounded:
species effects cannot be distinguished from location effects.

In [ ]:
def _heatmap(ax, mat: pd.DataFrame, title: str,
             cmap: str = 'YlOrRd', rank_label: bool = True,
             row_order=None, fontsize_cell: int = 7):
    """Plot a count matrix as a heatmap with optional SVD rank annotation."""
    if row_order is not None:
        mat = mat.reindex(row_order)
    vals = mat.fillna(0).values.astype(float)
    im = ax.imshow(vals, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.75, label='Sample count')
    ax.set_xticks(range(mat.shape[1]))
    ax.set_yticks(range(mat.shape[0]))
    ax.set_xticklabels(mat.columns, rotation=45, ha='right', fontsize=7.5)
    ax.set_yticklabels(mat.index, fontsize=8)
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = int(vals[i, j])
            if v > 0:
                ax.text(j, i, str(v), ha='center', va='center',
                        fontsize=fontsize_cell, color='black')
    if rank_label and vals.max() > 0:
        _, s, _ = np.linalg.svd(vals / (vals.sum() + 1e-9))
        explained = s[0]**2 / (s**2).sum()
        ax.set_title(f'{title}\n(top SV explains {explained:.0%} of variance)',
                     fontsize=9, fontweight='bold')
    else:
        ax.set_title(title, fontsize=9, fontweight='bold')


# ── species × country ────────────────────────────────────────────────────
mat_country = df.pivot_table(index='species_name', columns='country_code',
                              values='doy', aggfunc='count', fill_value=0)

# ── species × latitude bin ────────────────────────────────────────────────
mat_lat = df.pivot_table(index='species_name', columns='lat_bin',
                          values='doy', aggfunc='count', fill_value=0)

# ── species × year (all years) ────────────────────────────────────────────
mat_year = df.pivot_table(index='species_name', columns='year',
                           values='doy', aggfunc='count', fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
fig.suptitle('Cross-factor sample coverage  (rank-1 → confounded)',
             fontsize=12, fontweight='bold')

_heatmap(axes[0], mat_country, 'species × country', row_order=species_order, fontsize_cell=6)
axes[0].set_xlabel('Country code', fontsize=9)
axes[0].set_ylabel('Species (ordered by median bloom DOY)', fontsize=9)

_heatmap(axes[1], mat_lat, 'species × latitude band', row_order=species_order)
axes[1].set_xlabel('Latitude band', fontsize=9)
axes[1].set_ylabel('Species', fontsize=9)

plt.tight_layout()
plt.show()

# ── species × year (full heatmap) ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 4))
_heatmap(ax, mat_year, 'species × year', rank_label=False,
         row_order=species_order, fontsize_cell=5)
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('Species', fontsize=9)
plt.tight_layout()
plt.show()

# ── numeric rank diagnostics ──────────────────────────────────────────────
print('Singular value spectra (normalised):')
for name, mat in [('species × country', mat_country),
                   ('species × latitude', mat_lat)]:
    vals = mat.fillna(0).values.astype(float)
    if vals.max() == 0:
        continue
    _, s, _ = np.linalg.svd(vals / vals.sum())
    total = (s**2).sum()
    print(f'  {name}:')
    for i, sv in enumerate(s[:min(4, len(s))]):
        print(f'    SV{i+1}: {sv**2/total:.1%}')


## 4. Bloom DOY distributions

Distribution of first-bloom day-of-year per species (ordered by median DOY).
Narrow, non-overlapping distributions ease parameter estimation;
species with very few observations or no location overlap with others are
harder to include in a shared-parameter model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bloom DOY distributions (BBCH_60)', fontsize=12, fontweight='bold')

# — violin ————————————————————————————————————————————————————————————
ax = axes[0]
doy_data = [df[df['species_name'] == sp]['doy'].values for sp in species_order]
parts = ax.violinplot(doy_data, positions=range(len(species_order)),
                      showmedians=True, showextrema=False)
for pc, sp in zip(parts['bodies'], species_order):
    pc.set_facecolor(palette[sp])
    pc.set_alpha(0.65)
parts['cmedians'].set_color('black')

# DOY → month label helper
_DOY_MONTH = [(1,'Jan'),(32,'Feb'),(60,'Mar'),(91,'Apr'),(121,'May'),(152,'Jun')]
for doy, month in _DOY_MONTH:
    ax.axhline(doy, color='lightgrey', lw=0.7, zorder=0)
    ax.text(-0.6, doy, month, va='center', ha='right', fontsize=7, color='grey')

ax.set_xticks(range(len(species_order)))
ax.set_xticklabels([sp.split()[-1] for sp in species_order],
                    rotation=30, ha='right', fontsize=8.5)
ax.set_ylabel('Day of year (BBCH_60)', fontsize=9)
ax.set_title('Bloom DOY violin', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — sample count bar ——————————————————————————————————————————————————
ax = axes[1]
counts = df.groupby('species_name')['doy'].count().reindex(species_order)
ax.barh(range(len(species_order)), counts.values,
        color=[palette[sp] for sp in species_order], alpha=0.85)
ax.set_yticks(range(len(species_order)))
ax.set_yticklabels(species_order, fontsize=9)
ax.set_xlabel('Number of observations', fontsize=9)
ax.set_title('Observation count per species', fontsize=9, fontweight='bold')
for i, v in enumerate(counts.values):
    ax.text(v + 30, i, str(v), va='center', fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


## 5. Climate range per species

For a phenology model to be identifiable, each species must be observed across
a range of **temperatures, chilling accumulations, and spring forcing**.
Narrow distributions signal insufficient driver variation to constrain parameters;
species with non-overlapping climate ranges are harder to include in a
shared-parameter model.

- **Mean winter temperature (Oct–Jan)**: primary driver of chilling accumulation
- **Total chill days (T ≤ 7.2 °C)**: classic chilling unit
- **Forcing GDU (T > 4 °C)**: spring forcing accumulation

In [ ]:
if not HAS_TEMP:
    print('No temperature data — set DOWNLOAD_TEMPERATURE = True and re-run.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('CF-model driver distributions by species',
                 fontsize=12, fontweight='bold')

    plot_vars = [
        ('mean_winter_T', 'Mean winter temperature (°C)', 'Oct–Jan mean T'),
        ('chill_days',    'Total chill days (T ≤ 7.2 °C)', 'Season chill days'),
        ('forcing_gdu',   'Total forcing GDU (T > 4 °C)',  'Season forcing GDU'),
    ]

    for ax, (col, xlabel, title) in zip(axes, plot_vars):
        data = [df[df['species_name'] == sp][col].dropna().values
                for sp in species_order]
        bp = ax.boxplot(data, patch_artist=True,
                        medianprops=dict(color='black', lw=1.8),
                        whiskerprops=dict(lw=1), capprops=dict(lw=1),
                        flierprops=dict(marker='o', markersize=2, alpha=0.3,
                                        markeredgewidth=0))
        for patch, sp in zip(bp['boxes'], species_order):
            patch.set_facecolor(palette[sp])
            patch.set_alpha(0.75)

        # Overlay all-species IQR as a grey band
        all_vals = df[col].dropna()
        ax.axhspan(all_vals.quantile(0.25), all_vals.quantile(0.75),
                   color='lightgrey', alpha=0.25, zorder=0,
                   label='All-species IQR')

        ax.set_xticks(range(1, len(species_order) + 1))
        ax.set_xticklabels([s.split()[-1] for s in species_order],
                            rotation=30, ha='right', fontsize=8.5)
        ax.set_xlabel('Species', fontsize=9)
        ax.set_ylabel(xlabel, fontsize=9)
        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[0].legend(fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.show()

    # ── Print ranges ────────────────────────────────────────────────────
    print('Climate ranges per species (IQR):')
    for col, label in [('mean_winter_T', 'Winter T (°C)'),
                        ('chill_days',    'Chill days  '),
                        ('forcing_gdu',   'Forcing GDU ')]:
        print(f'  {label}:')
        for sp in species_order:
            vals = df[df['species_name'] == sp][col].dropna()
            if vals.empty:
                continue
            q25, q75 = vals.quantile([0.25, 0.75])
            print(f'    {sp:30s}  '
                  f'IQR [{q25:+6.1f}, {q75:+6.1f}]  '
                  f'range [{vals.min():+6.1f}, {vals.max():+6.1f}]')


## 5b. Species × climate bin

Three complementary views of climate coverage:

1. **Species × winter T bin** — chilling driver (Oct–Jan)
2. **Species × spring T bin** — forcing driver (Feb–Apr)
3. **2D climate space** — winter T × spring T scatter coloured by species, showing whether species occupy distinct or overlapping climate niches in both dimensions simultaneously

Nearly rank-1 heatmaps, or tightly clustered non-overlapping clouds in the scatter, signal that species and climate are confounded.

In [ ]:
if not HAS_TEMP:
    print('No temperature data — set DOWNLOAD_TEMPERATURE = True and re-run.')
else:
    df['climate_bin'] = pd.cut(df['mean_winter_T'],
                                bins=CLIMATE_BINS, labels=CLIMATE_LABELS)
    df['spring_bin']  = pd.cut(df['mean_spring_T'],
                                bins=SPRING_BINS,  labels=SPRING_LABELS)

    mat_climate = df.pivot_table(index='species_name', columns='climate_bin',
                                  values='doy', aggfunc='count', fill_value=0)
    mat_spring  = df.pivot_table(index='species_name', columns='spring_bin',
                                  values='doy', aggfunc='count', fill_value=0)

    # ── Heatmaps: winter T and spring T ──────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(20, 5.5))
    fig.suptitle('Species × climate bin coverage  (rank-1 → confounded)',
                 fontsize=12, fontweight='bold')

    _heatmap(axes[0], mat_climate, 'species × winter T bin (Oct–Jan)',
             row_order=species_order)
    axes[0].set_xlabel('Mean winter temperature bin (°C)', fontsize=9)
    axes[0].set_ylabel('Species (ordered by median bloom DOY)', fontsize=9)

    _heatmap(axes[1], mat_spring, 'species × spring T bin (Feb–Apr)',
             row_order=species_order)
    axes[1].set_xlabel('Mean spring temperature bin (°C)', fontsize=9)
    axes[1].set_ylabel('', fontsize=9)

    plt.tight_layout()
    plt.show()

    # ── 2D climate space: winter T × spring T coloured by species ────────
    fig, ax = plt.subplots(figsize=(9, 6.5))
    fig.suptitle('2D climate niche — winter T × spring T',
                 fontsize=12, fontweight='bold')

    for sp in species_order:
        sub = df[df['species_name'] == sp].dropna(
            subset=['mean_winter_T', 'mean_spring_T'])
        sub = sub.sample(min(400, len(sub)), random_state=0)
        ax.scatter(sub['mean_winter_T'], sub['mean_spring_T'],
                   c=[palette[sp]], s=10, alpha=0.35, label=sp)

    # Overlay per-species 50% and 90% ellipses
    from matplotlib.patches import Ellipse
    import matplotlib.transforms as transforms
    for sp in species_order:
        sub = df[df['species_name'] == sp].dropna(
            subset=['mean_winter_T', 'mean_spring_T'])
        if len(sub) < 10:
            continue
        x, y = sub['mean_winter_T'].values, sub['mean_spring_T'].values
        cov  = np.cov(x, y)
        vals_ev, vecs = np.linalg.eigh(cov)
        order = vals_ev.argsort()[::-1]
        vals_ev, vecs = vals_ev[order], vecs[:, order]
        angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        # 90% ellipse (chi2 2-dof 90% ≈ 4.605)
        w90, h90 = 2 * np.sqrt(4.605 * vals_ev)
        ell = Ellipse((x.mean(), y.mean()), w90, h90, angle=angle,
                      edgecolor=palette[sp], facecolor='none',
                      lw=1.5, ls='--', alpha=0.8)
        ax.add_patch(ell)

    ax.set_xlabel('Mean winter temperature (Oct–Jan, °C)', fontsize=10)
    ax.set_ylabel('Mean spring temperature (Feb–Apr, °C)', fontsize=10)
    ax.legend(fontsize=8, ncol=2, framealpha=0.9,
              title='Species  (dashed ellipse = 90% of obs)')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

    # ── SVD rank diagnostics ──────────────────────────────────────────────
    print('SV1 of coverage matrices (100% = perfectly rank-1 = confounded):')
    for name, mat in [('species × winter T', mat_climate),
                       ('species × spring T', mat_spring)]:
        vals = mat.fillna(0).values.astype(float)
        _, s, _ = np.linalg.svd(vals / vals.sum())
        total = (s**2).sum()
        print(f'  {name}: SV1={s[0]**2/total:.0%}  '
              f'SV2={s[1]**2/total:.0%}  SV3={s[2]**2/total:.0%}')


## 5c. Driver range — chilling and forcing space

Scatter of chilling vs forcing per observation, coloured by species.
A good dataset covers a wide region of the (chill, forcing) plane;
tight clustering means only one climate regime is represented and
the model's threshold parameters cannot be reliably estimated.

In [ ]:
if not HAS_TEMP:
    print('No temperature data — set DOWNLOAD_TEMPERATURE = True and re-run.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    fig.suptitle('Driver range analysis — identifiability of model parameters',
                 fontsize=12, fontweight='bold')

    # — chilling vs forcing scatter ─────────────────────────────────────
    ax = axes[0]
    for sp in species_order:
        sub = df[df['species_name'] == sp].dropna(subset=['chill_days', 'forcing_gdu'])
        sub = sub.sample(min(300, len(sub)), random_state=0)
        ax.scatter(sub['chill_days'], sub['forcing_gdu'],
                   c=[palette[sp]], s=8, alpha=0.35, label=sp)
    ax.set_xlabel('Total chill days (T ≤ 7.2 °C)', fontsize=9)
    ax.set_ylabel('Total forcing GDU (T > 4 °C)', fontsize=9)
    ax.set_title('Chilling vs forcing driver space', fontsize=9, fontweight='bold')
    ax.legend(fontsize=7, ncol=2, framealpha=0.9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # — chill days vs bloom DOY ─────────────────────────────────────────
    ax = axes[1]
    for i, sp in enumerate(species_order):
        sub = df[df['species_name'] == sp].dropna(subset=['chill_days', 'doy'])
        sub = sub.sample(min(300, len(sub)), random_state=0)
        ax.scatter(sub['chill_days'], sub['doy'],
                   c=[palette[sp]], s=8, alpha=0.35, label=sp)
        r = sub[['chill_days', 'doy']].corr().iloc[0, 1]
        ax.annotate(f'r={r:.2f}', xy=(0.03, 0.96 - i * 0.07),
                    xycoords='axes fraction', fontsize=7,
                    color=palette[sp])
    ax.set_xlabel('Total chill days', fontsize=9)
    ax.set_ylabel('Bloom DOY', fontsize=9)
    ax.set_title('Chill days vs bloom date', fontsize=9, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # — forcing GDU vs bloom DOY ─────────────────────────────────────────
    ax = axes[2]
    for i, sp in enumerate(species_order):
        sub = df[df['species_name'] == sp].dropna(subset=['forcing_gdu', 'doy'])
        sub = sub.sample(min(300, len(sub)), random_state=0)
        ax.scatter(sub['forcing_gdu'], sub['doy'],
                   c=[palette[sp]], s=8, alpha=0.35, label=sp)
        r = sub[['forcing_gdu', 'doy']].corr().iloc[0, 1]
        ax.annotate(f'r={r:.2f}', xy=(0.03, 0.96 - i * 0.07),
                    xycoords='axes fraction', fontsize=7,
                    color=palette[sp])
    ax.set_xlabel('Total forcing GDU', fontsize=9)
    ax.set_ylabel('Bloom DOY', fontsize=9)
    ax.set_title('Forcing GDU vs bloom date', fontsize=9, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

    print('Driver ranges by species:')
    for col, label in [('chill_days', 'Chill days'), ('forcing_gdu', 'Forcing GDU')]:
        print(f'  {label}:')
        for sp in species_order:
            sub = df[df['species_name'] == sp][col].dropna()
            if sub.empty:
                continue
            print(f'    {sp:30s}  {sub.min():.0f} – {sub.max():.0f}  '
                  f'(mean {sub.mean():.0f}, std {sub.std():.0f})')


## 6. Shared-location coverage

Parameter estimation is most reliable when the same **physical station** records multiple species (controls for site climate). The matrix below shows, for each pair of species, the number of locations where both species have been observed. Zeros mean those two species can only be compared indirectly.

In [ ]:
# ── Locations per species ────────────────────────────────────────────────
sp_locs = (
    df.groupby(['loc_id', 'species_name'])
      .size()
      .unstack(fill_value=0)
      .clip(upper=1)   # presence/absence
      .reindex(columns=species_order, fill_value=0)
)

# Pairwise shared-location count
n_sp = len(species_order)
shared_mat = np.zeros((n_sp, n_sp), dtype=int)
for i in range(n_sp):
    for j in range(n_sp):
        both = (sp_locs.iloc[:, i] > 0) & (sp_locs.iloc[:, j] > 0)
        shared_mat[i, j] = both.sum()

shared_df = pd.DataFrame(shared_mat, index=species_order, columns=species_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Shared-location coverage', fontsize=12, fontweight='bold')

# — shared-location matrix ————————————————————————————————————————————
ax = axes[0]
im = ax.imshow(shared_mat, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='N locations with both species')
ax.set_xticks(range(n_sp))
ax.set_yticks(range(n_sp))
ax.set_xticklabels([s.split()[-1] for s in species_order],
                    rotation=40, ha='right', fontsize=8)
ax.set_yticklabels(species_order, fontsize=8)
for i in range(n_sp):
    for j in range(n_sp):
        v = shared_mat[i, j]
        color = 'white' if v > shared_mat.max() * 0.6 else 'black'
        ax.text(j, i, str(v), ha='center', va='center', fontsize=7.5, color=color)
ax.set_title('Pairwise shared-location count\n'
             '(diagonal = n_locations with that species)',
             fontsize=9, fontweight='bold')

# — histogram of n_species per location ———————————————————————————————
ax = axes[1]
n_sp_per_loc = sp_locs.sum(axis=1)
counts_sp = n_sp_per_loc.value_counts().sort_index()
ax.bar(counts_sp.index.astype(str), counts_sp.values,
       color='steelblue', alpha=0.8, edgecolor='white')
ax.set_xlabel('Number of species present at location', fontsize=9)
ax.set_ylabel('Number of locations', fontsize=9)
ax.set_title('How many species co-occur at the same station?',
             fontsize=9, fontweight='bold')
for x, v in zip(counts_sp.index, counts_sp.values):
    ax.text(str(x), v + 5, str(v), ha='center', fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f'Locations with ≥ 2 species: '
      f'{(n_sp_per_loc >= 2).sum()} / {len(n_sp_per_loc)} '
      f'({(n_sp_per_loc >= 2).mean():.0%})')
print(f'Locations with all {n_sp} species: {(n_sp_per_loc == n_sp).sum()}')


## 7. Temporal variation

Multiple years are only useful if they span meaningfully different conditions.
- **Bloom trends**: do species track each other over time (consistent year effects)?
- **Year anomalies**: are there early and late years spread across the record?
- **Species × year coverage**: are all species measured in each year?

In [ ]:
# Per-year, per-species median bloom DOY
yr_sp = (
    df.groupby(['year', 'species_name'])['doy']
      .median()
      .reset_index()
)

# Cross-species mean bloom anomaly per year
yr_mean = df.groupby('year')['doy'].median().reset_index()
yr_mean['anom'] = yr_mean['doy'] - yr_mean['doy'].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Temporal coverage and variation', fontsize=12, fontweight='bold')

# — bloom trend by species ————————————————————————————————————————————
ax = axes[0, 0]
for sp in species_order:
    sub = yr_sp[yr_sp['species_name'] == sp]
    if sub.empty:
        continue
    rolled = sub.set_index('year')['doy'].rolling(5, center=True, min_periods=2).median()
    ax.plot(rolled.index, rolled.values, color=palette[sp], lw=1.7, label=sp)
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('Median bloom DOY (5-yr rolling)', fontsize=9)
ax.set_title('Bloom date trends by species', fontsize=9, fontweight='bold')
ax.legend(fontsize=7, ncol=2, framealpha=0.9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — annual bloom anomaly ——————————————————————————————————————————————
ax = axes[0, 1]
bar_colors = ['#e74c3c' if v > 0 else '#2980b9' for v in yr_mean['anom']]
ax.bar(yr_mean['year'], yr_mean['anom'], color=bar_colors, alpha=0.8)
ax.axhline(0, color='grey', lw=0.8)
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('Bloom DOY anomaly (days)', fontsize=9)
ax.set_title('Cross-species bloom anomaly per year\n'
             '(red = late year, blue = early year)',
             fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — sample count per year ——————————————————————————————————————————————
ax = axes[1, 0]
yr_counts = df.groupby(['year', 'species_name'])['doy'].count().unstack(fill_value=0)
yr_counts = yr_counts.reindex(columns=species_order, fill_value=0)
bottom = np.zeros(len(yr_counts))
for sp in species_order:
    vals = yr_counts[sp].values
    ax.bar(yr_counts.index, vals, bottom=bottom,
           color=palette[sp], label=sp, alpha=0.85, width=0.9)
    bottom += vals
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('Observation count', fontsize=9)
ax.set_title('Sample count per year (stacked by species)',
             fontsize=9, fontweight='bold')
ax.legend(fontsize=7, ncol=2, framealpha=0.9, loc='upper left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — interannual DOY spread: how much year-to-year variation? ——————————
ax = axes[1, 1]
# Per-species, per-location interannual standard deviation
iav = (
    df.groupby(['species_name', 'loc_id'])['doy']
      .std()
      .reset_index(name='iav')
)
iav_data = [iav[iav['species_name'] == sp]['iav'].dropna().values
            for sp in species_order]
bp = ax.boxplot(iav_data, patch_artist=True,
                medianprops=dict(color='black', lw=1.5),
                whiskerprops=dict(lw=1),
                capprops=dict(lw=1))
for patch, sp in zip(bp['boxes'], species_order):
    patch.set_facecolor(palette[sp])
    patch.set_alpha(0.7)
ax.set_xticks(range(1, len(species_order) + 1))
ax.set_xticklabels([s.split()[-1] for s in species_order],
                    rotation=30, ha='right', fontsize=8.5)
ax.set_ylabel('Interannual std of bloom DOY (days)', fontsize=9)
ax.set_title('Interannual variability per station\n'
             '(larger = more signal for year-effect estimation)',
             fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

q33, q67 = yr_mean['anom'].quantile([0.33, 0.67])
early_years = yr_mean[yr_mean['anom'] < q33]['year'].tolist()
late_years  = yr_mean[yr_mean['anom'] > q67]['year'].tolist()
print(f'Early-bloom years (bottom third): {sorted(early_years)}')
print(f'Late-bloom  years (top third):    {sorted(late_years)}')
print(f'Bloom DOY range: {df["doy"].min()} – {df["doy"].max()} '
      f'(spread {df["doy"].max() - df["doy"].min()} days)')


## 8. Bloom DOY vs latitude

Latitude is a proxy for mean climate. If species cluster in different latitude bands (§3) and bloom DOY is strongly correlated with latitude, then species effects on bloom timing are confounded with latitude (climate). We check both the strength of the latitude–bloom relationship and whether it is similar across species.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Bloom DOY vs latitude — latitudinal confounding check',
             fontsize=12, fontweight='bold')

# — raw scatter ———————————————————————————————————————————————————————
ax = axes[0]
for sp in species_order:
    sub = df[df['species_name'] == sp].sample(min(500, len(df[df['species_name']==sp])),
                                               random_state=0)
    ax.scatter(sub['lat'], sub['doy'], c=[palette[sp]], s=6,
               alpha=0.35, label=sp)
ax.set_xlabel('Latitude (°N)', fontsize=9)
ax.set_ylabel('Bloom DOY', fontsize=9)
ax.set_title('Bloom DOY vs latitude (up to 500 pts per species)',
             fontsize=9, fontweight='bold')
ax.legend(fontsize=7, ncol=2, framealpha=0.9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — slope and r per species ————————————————————————————————————————————
ax = axes[1]
r_vals, slope_vals = [], []
for sp in species_order:
    sub = df[df['species_name'] == sp][['lat', 'doy']].dropna()
    if len(sub) < 5:
        r_vals.append(np.nan)
        slope_vals.append(np.nan)
        continue
    r = sub.corr().iloc[0, 1]
    coef = np.polyfit(sub['lat'], sub['doy'], 1)
    r_vals.append(r)
    slope_vals.append(coef[0])

x = range(len(species_order))
ax2 = ax.twinx()
ax.bar(x, r_vals, color=[palette[sp] for sp in species_order], alpha=0.8,
       label='Pearson r')
ax2.plot(x, slope_vals, 'ko-', ms=5, lw=1.5, label='Slope (days/°N)')
ax.axhline(0, color='grey', lw=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels([s.split()[-1] for s in species_order],
                    rotation=30, ha='right', fontsize=8.5)
ax.set_ylabel('Pearson r  (lat vs DOY)', fontsize=9)
ax2.set_ylabel('Slope  (days per °N)', fontsize=9)
ax.set_title('Latitude–bloom correlation and slope per species',
             fontsize=9, fontweight='bold')
ax.set_ylim(-1.05, 1.05)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, framealpha=0.9)
ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.show()

print('Latitude–bloom Pearson r per species:')
for sp, r in zip(species_order, r_vals):
    print(f'  {sp:30s}  r = {r:+.3f}')


## 9. Summary: adequacy verdict

Collate the key diagnostics into a single table.

In [ ]:
def _sv1(mat):
    vals = mat.fillna(0).values.astype(float)
    if vals.max() == 0:
        return float('nan')
    _, s, _ = np.linalg.svd(vals / vals.sum())
    return float(s[0]**2 / (s**2).sum())

n_total    = len(df)
n_locs     = df['loc_id'].nunique()
n_years    = df['year'].nunique()
year_range = df['year'].max() - df['year'].min()
frac_shared = (n_sp_per_loc >= 2).mean()
bloom_std   = df['doy'].std()
sv1_country = _sv1(mat_country)
sv1_lat     = _sv1(mat_lat)

n_obs_per_sp = df.groupby('species_name')['doy'].count().reindex(species_order)
min_obs_sp   = n_obs_per_sp.min()
min_obs_name = n_obs_per_sp.idxmin()

rows = [
    ('Total observations',         f'{n_total:,}',           'Should be >> n_params'),
    ('Locations',                  f'{n_locs:,}',            '—'),
    ('Years',                      f'{n_years}  ({year_range} yr span)',
                                                              'Need interannual variation'),
    ('Species',                    str(n_sp),                '—'),
    ('Smallest species',           f'{min_obs_name}: {min_obs_sp:,} obs',
                                                              'Need ≥ ~200 obs to constrain parameters'),
    ('Species × country  SV1',     f'{sv1_country:.0%}',    '< 80% adequate; ≥ 90% confounded'),
    ('Species × latitude SV1',     f'{sv1_lat:.0%}',        '< 80% adequate; ≥ 90% confounded'),
    ('Locations with ≥ 2 species', f'{frac_shared:.0%}',    '> 40% recommended for cross-species estimation'),
    ('Bloom DOY std (all)',         f'{bloom_std:.1f} days', 'Larger = more phenological signal'),
]

if HAS_TEMP:
    sv1_climate = _sv1(mat_climate)
    wT_range  = df['mean_winter_T'].max() - df['mean_winter_T'].min()
    chill_range = df['chill_days'].max()  - df['chill_days'].min()
    rows += [
        ('Species × climate SV1',      f'{sv1_climate:.0%}',
                                        '< 80% adequate; ≥ 90% confounded'),
        ('Winter T range (all)',        f'{df["mean_winter_T"].min():.1f} – '
                                        f'{df["mean_winter_T"].max():.1f} °C  '
                                        f'(spread {wT_range:.1f} °C)',
                                        '> 8 °C recommended for CF models'),
        ('Chill days range (all)',      f'{df["chill_days"].min():.0f} – '
                                        f'{df["chill_days"].max():.0f}  '
                                        f'(spread {chill_range:.0f})',
                                        'Wide range constrains threshold_c'),
    ]

display(pd.DataFrame(rows, columns=['Diagnostic', 'Value', 'Guideline']))
